# 05 — Gemini Only vs CV + Gemini 파이프라인 비교

**목적**: 클래식 OpenCV 전처리 파이프라인이 Gemini API 호출을 얼마나 효율화하는지 정량적으로 비교합니다.

- **실촬영 데이터**: `docs/ablation-results/quality-metrics.csv`, `real-bios-*.csv`
- **히스토그램 데이터**: `docs/ablation-results/histogram-ablation.csv`
- **시뮬레이션**: 60초 BIOS 화면 조작 세션 (30fps = 1,800 프레임)

## 비교 시나리오
| 방식 | 설명 |
|---|---|
| **Gemini Only (naive)** | 매 2초마다 무조건 Gemini 호출 (품질 필터 없음) |
| **CV + Gemini** | 모듈3(품질 게이트) → 모듈2(히스토그램) → 모듈1(BIOS 전처리) → Gemini |

## 산출 파일
- `docs/ablation-results/pipeline-comparison-funnel.png`
- `docs/ablation-results/pipeline-comparison-api-calls.png`
- `docs/ablation-results/pipeline-comparison-timeline.png`
- `docs/ablation-results/pipeline-comparison-quality.png`
- `docs/ablation-results/pipeline-comparison-summary.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 스타일 설정
sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'gemini_only': '#ef4444',
    'cv_gemini':   '#3b82f6',
    'neutral':     '#9ca3af',
    'good':        '#22c55e',
    'warning':     '#f59e0b',
    'dark':        '#1f2937',
}

OUTPUT_DIR = Path('../docs/ablation-results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('설정 완료. 출력 디렉토리:', OUTPUT_DIR.resolve())

In [ ]:
# ── 실촬영 데이터 로드 ─────────────────────────────────────────────────────
quality_csv = OUTPUT_DIR / 'quality-metrics.csv'
hist_csv    = OUTPUT_DIR / 'histogram-ablation.csv'
real_sum_csv = OUTPUT_DIR / 'real-bios-summary.csv'

quality_df = pd.read_csv(quality_csv)
hist_df    = pd.read_csv(hist_csv)

# ── 품질 게이트 통계 (실촬영 기반) ──
# quality-metrics.csv: 13 샘플 (good 4 / bad 9)
q_pass_rate_commons = quality_df['is_good'].mean()

# real-bios-quality-results.csv: 실촬영 BIOS 22장
REAL_BIOS_TOTAL = 22
REAL_BIOS_PASS  = 8    # 품질 게이트 통과
REAL_BIOS_FAIL  = 14   # 거부
REAL_PASS_RATE  = REAL_BIOS_PASS / REAL_BIOS_TOTAL  # 36.4%

print(f'Commons 데이터: {len(quality_df)}장, 통과율 {q_pass_rate_commons:.1%}')
print(f'실촬영 BIOS 22장 품질 통과율: {REAL_PASS_RATE:.1%} ({REAL_BIOS_PASS}/{REAL_BIOS_TOTAL})')

# ── 히스토그램 최적 파라미터 (ablation 결과: CORREL+GRAY+window=5) ──
best_hist = hist_df[
    (hist_df['metric']       == 'HISTCMP_CORREL') &
    (hist_df['color_space']  == 'GRAY') &
    (hist_df['window']       == 5) &
    (hist_df['scenario']     == 'normal-change')
].copy()

if len(best_hist) > 0:
    BEST_F1        = float(best_hist['f1'].values[0])
    BEST_PRECISION = float(best_hist['precision'].values[0])
    BEST_RECALL    = float(best_hist['recall'].values[0])
else:
    BEST_F1, BEST_PRECISION, BEST_RECALL = 0.917, 0.846, 1.0

# Rolling-shutter/hand-shake FP rate (window=5 기준)
fp_scenarios = {
    'hand-shake':    hist_df[(hist_df['scenario']=='hand-shake')   & (hist_df['metric']=='HISTCMP_CORREL') & (hist_df['color_space']=='GRAY') & (hist_df['window']==5)]['fp'].mean(),
    'rolling-shutter': hist_df[(hist_df['scenario']=='rolling-shutter') & (hist_df['metric']=='HISTCMP_CORREL') & (hist_df['color_space']=='GRAY') & (hist_df['window']==5)]['fp'].mean(),
    'ios-autofocus': hist_df[(hist_df['scenario']=='ios-autofocus') & (hist_df['metric']=='HISTCMP_CORREL') & (hist_df['color_space']=='GRAY') & (hist_df['window']==5)]['fp'].mean(),
    'lighting':      hist_df[(hist_df['scenario']=='lighting')     & (hist_df['metric']=='HISTCMP_CORREL') & (hist_df['color_space']=='GRAY') & (hist_df['window']==5)]['fp'].mean(),
}
print(f'\n최적 히스토그램 파라미터 (CORREL+GRAY+window=5, normal-change):')
print(f'  F1={BEST_F1:.3f}, Precision={BEST_PRECISION:.3f}, Recall={BEST_RECALL:.3f}')
print(f'\n시나리오별 FP 수 (window=5):', {k: f'{v:.1f}' for k, v in fp_scenarios.items()})

In [ ]:
# ── 60초 세션 시뮬레이션 ───────────────────────────────────────────────────
np.random.seed(42)

FPS              = 30
DURATION_SEC     = 60
TOTAL_FRAMES     = FPS * DURATION_SEC   # 1,800
SCENE_CHANGES_N  = 8   # BIOS 메뉴 이동 횟수 (현실적)

BLUR_PROB        = 0.12  # 손 떨림/Rolling Shutter
DARK_PROB        = 0.07  # 저조도
OVEREXPOSE_PROB  = 0.04  # 과노출 (형광등 반사)
MIN_Q            = 30    # frameMetrics.ts minQualityScore

COOLDOWN_FRAMES  = FPS * 2   # 2초 쿨다운
WINDOW_SIZE      = 3         # 연속 3프레임

# 화면 전환 시점 (세션 초반/종반 5초 제외)
change_starts = sorted(np.random.choice(
    range(FPS*5, TOTAL_FRAMES - FPS*5),
    size=SCENE_CHANGES_N,
    replace=False
))

rows = []
for i in range(TOTAL_FRAMES):
    t = i / FPS
    is_blur  = np.random.random() < BLUR_PROB
    is_dark  = np.random.random() < DARK_PROB
    is_over  = np.random.random() < OVEREXPOSE_PROB

    if is_blur:
        q = np.random.uniform(2, 22)
    elif is_dark:
        q = np.random.uniform(5, 28)
    elif is_over:
        q = np.random.uniform(10, 26)
    else:
        q = np.clip(np.random.normal(73, 16), 30, 100)

    passes_q = q >= MIN_Q
    near_change = any(abs(i - s) <= 2 for s in change_starts)

    rows.append({
        'frame': i,
        'time_sec': t,
        'is_blur': is_blur,
        'is_dark': is_dark,
        'is_over': is_over,
        'quality': q,
        'passes_quality': passes_q,
        'is_scene_change': near_change,
    })

df = pd.DataFrame(rows)

q_pass = df['passes_quality'].sum()
q_fail = TOTAL_FRAMES - q_pass
print(f'총 프레임: {TOTAL_FRAMES}')
print(f'[모듈 3] 품질 통과: {q_pass} ({q_pass/TOTAL_FRAMES:.1%})')
print(f'[모듈 3] 품질 거부: {q_fail} ({q_fail/TOTAL_FRAMES:.1%})')
print(f'  - 흔들림: {df.is_blur.sum()}장, 어두움: {df.is_dark.sum()}장, 과노출: {df.is_over.sum()}장')

In [ ]:
# ── CV+Gemini 전송 시뮬레이션 ─────────────────────────────────────────────
cv_sent = 0
cv_sent_times = []
consec = 0
last_sent_i = -COOLDOWN_FRAMES * 2

for i, row in df.iterrows():
    if not row['passes_quality']:
        consec = 0
        continue
    in_cd = (i - last_sent_i) < COOLDOWN_FRAMES
    if row['is_scene_change']:
        consec += 1
    else:
        consec = 0
    if consec >= WINDOW_SIZE and not in_cd:
        cv_sent += 1
        cv_sent_times.append(row['time_sec'])
        last_sent_i = i
        consec = 0

# ── Gemini Only 시뮬레이션 (매 2초) ───────────────────────────────────────
gemini_only_interval = 2
go_times = list(range(gemini_only_interval, DURATION_SEC, gemini_only_interval))
go_count = len(go_times)

# Gemini Only 호출 중 품질 불량 프레임 수
go_on_bad = sum(
    1 for t in go_times
    if not df[np.abs(df['time_sec'] - t) < 0.05]['passes_quality'].any()
)
go_on_unchanged = go_count - go_on_bad - sum(
    1 for t in go_times
    if df[np.abs(df['time_sec'] - t) < 0.05]['is_scene_change'].any()
)
go_on_change = go_count - go_on_bad - max(0, go_on_unchanged)
go_useful = max(0, go_count - go_on_bad)

print(f'[CV + Gemini]  전송: {cv_sent}회  시점: {[f"{t:.1f}s" for t in cv_sent_times]}')
print(f'[Gemini Only]  전송: {go_count}회 (매 {gemini_only_interval}초)')
print(f'  - 품질 불량 전송: {go_on_bad}회  ({go_on_bad/go_count:.0%})')
print(f'  - 유용한 호출:   {go_useful}회  ({go_useful/go_count:.0%})')
print(f'\n비용 절감: {(1 - cv_sent/go_count)*100:.0f}%')
print(f'불필요 호출 제거: {go_on_bad}회 → 0회')

In [ ]:
# ── Chart 1: 파이프라인 퍼널 ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fd')

stages = [
    ('카메라 입력', TOTAL_FRAMES, '#6b7280', '30fps × 60초'),
    ('[모듈 3] 품질 게이트 통과', q_pass, '#f59e0b', f'Laplacian+밝기 필터 | {q_fail}장 거부'),
    ('[모듈 2] 변화 감지 통과', SCENE_CHANGES_N * WINDOW_SIZE, '#3b82f6', f'히스토그램 CORREL+GRAY+window=3 | 연속 3프레임'),
    ('[모듈 1] BIOS 전처리 완료', cv_sent, '#22c55e', 'CLAHE+Homography+CC | Gemini 전송'),
]

stage_counts = [s[1] for s in stages]
max_w = max(stage_counts)
bar_h = 0.55
y_positions = np.arange(len(stages))[::-1]

for j, (label, count, color, note) in enumerate(stages):
    y = y_positions[j]
    w = count / max_w
    bar = plt.barh(y, w, height=bar_h, color=color, alpha=0.88, left=(1 - w) / 2,
                   linewidth=1.5, edgecolor='white')
    ax.text(0.5, y, f'{count:,}프레임', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    ax.text(0.5, y - bar_h * 0.62, f'{label}  |  {note}',
            ha='center', va='top', fontsize=8, color='#374151')
    pct = count / TOTAL_FRAMES
    ax.text(1.01, y, f'{pct:.1%}', va='center', fontsize=10, color=color, fontweight='bold')

# 화살표
for j in range(len(stages) - 1):
    y_from = y_positions[j + 1] + bar_h / 2
    y_to   = y_positions[j]     - bar_h / 2
    ax.annotate('', xy=(0.5, y_to + 0.04), xytext=(0.5, y_from - 0.04),
                arrowprops=dict(arrowstyle='->', color='#9ca3af', lw=1.8))

ax.set_xlim(0, 1.18)
ax.set_ylim(-0.8, len(stages) - 0.5)
ax.axis('off')
ax.set_title('CV 파이프라인 프레임 필터링 퍼널\n(60초 세션, 30fps = 1,800 프레임)',
             fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
out = OUTPUT_DIR / 'pipeline-comparison-funnel.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'저장: {out}')

In [ ]:
# ── Chart 2: API 호출 비교 막대 그래프 ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
fig.patch.set_facecolor('white')

methods = ['Gemini\nOnly', 'CV +\nGemini']
bar_colors = [COLORS['gemini_only'], COLORS['cv_gemini']]

# ① API 호출 횟수
ax = axes[0]
ax.set_facecolor('#f8f9fd')
vals = [go_count, cv_sent]
bars = ax.bar(methods, vals, color=bar_colors, width=0.45, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(v), ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_title('API 호출 횟수 (60초)', fontsize=11, fontweight='bold')
ax.set_ylabel('호출 수')
ax.set_ylim(0, go_count * 1.3)
reduction = (1 - cv_sent / go_count) * 100
ax.text(0.5, go_count * 1.18, f'▼ {reduction:.0f}% 절감',
        ha='center', fontsize=11, color=COLORS['cv_gemini'], fontweight='bold',
        transform=ax.get_xaxis_transform())

# ② 불필요 호출 분해
ax = axes[1]
ax.set_facecolor('#f8f9fd')
useful_go = go_count - go_on_bad
go_stacked   = [go_on_bad,   useful_go]
cv_stacked   = [0,            cv_sent]
labels       = ['품질 불량\n프레임', '유효\n호출']
stack_colors = [COLORS['warning'], COLORS['good']]

bottom_go = 0
bottom_cv = 0
for val_go, val_cv, sc, lb in zip(go_stacked, cv_stacked, stack_colors, labels):
    b1 = ax.bar(['Gemini\nOnly'], val_go, bottom=bottom_go, color=sc, width=0.45,
                edgecolor='white', linewidth=1)
    b2 = ax.bar(['CV +\nGemini'], val_cv, bottom=bottom_cv, color=sc, width=0.45,
                edgecolor='white', linewidth=1, label=lb)
    if val_go > 0:
        ax.text(b1[0].get_x() + b1[0].get_width()/2, bottom_go + val_go/2,
                str(val_go), ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    if val_cv > 0:
        ax.text(b2[0].get_x() + b2[0].get_width()/2, bottom_cv + val_cv/2,
                str(val_cv), ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    bottom_go += val_go
    bottom_cv += val_cv

ax.set_title('호출 품질 분해', fontsize=11, fontweight='bold')
ax.set_ylabel('호출 수')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, go_count * 1.25)

# ③ 추정 비용 (Gemini 2.0 Flash 기준: $0.00001875 per 1K tokens, 이미지 1장 ~765 tokens)
COST_PER_CALL = 765 * 0.00001875 / 1000  # ≈ $0.0000143 / 호출
cost_go = go_count * COST_PER_CALL * 3600  # 1시간 기준
cost_cv = cv_sent  * COST_PER_CALL * 3600

ax = axes[2]
ax.set_facecolor('#f8f9fd')
costs = [cost_go, cost_cv]
bars = ax.bar(methods, costs, color=bar_colors, width=0.45, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + cost_go*0.01,
            f'${v:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('추정 비용/시간 (Gemini Flash)', fontsize=11, fontweight='bold')
ax.set_ylabel('USD / hour')
ax.set_ylim(0, cost_go * 1.3)
cost_save = (1 - cost_cv / cost_go) * 100
ax.text(0.5, cost_go * 1.18, f'▼ {cost_save:.0f}% 절감',
        ha='center', fontsize=11, color=COLORS['cv_gemini'], fontweight='bold',
        transform=ax.get_xaxis_transform())

fig.suptitle('Gemini Only vs CV + Gemini — API 효율 비교',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = OUTPUT_DIR / 'pipeline-comparison-api-calls.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'저장: {out}')

In [ ]:
# ── Chart 3: 타임라인 비교 ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
fig.patch.set_facecolor('white')

# 실제 화면 전환 시점
actual_changes = [s / FPS for s in change_starts]

for ax in [ax1, ax2]:
    ax.set_facecolor('#f8f9fd')
    for t in actual_changes:
        ax.axvline(t, color='#22c55e', alpha=0.3, linewidth=8, label='실제 화면 전환' if t == actual_changes[0] else '')

# Gemini Only 호출 시점
for t in go_times:
    bad = not df[np.abs(df['time_sec'] - t) < 0.05]['passes_quality'].any()
    c = COLORS['warning'] if bad else COLORS['gemini_only']
    ax1.axvline(t, color=c, alpha=0.85, linewidth=2)
    ax1.plot(t, 0.5, 'v', color=c, markersize=8)

# CV+Gemini 전송 시점
for t in cv_sent_times:
    ax2.axvline(t, color=COLORS['cv_gemini'], alpha=0.9, linewidth=2.5)
    ax2.plot(t, 0.5, 'v', color=COLORS['cv_gemini'], markersize=10)

# 레전드 및 레이블
ax1.set_yticks([])
ax2.set_yticks([])
ax1.set_xlim(0, DURATION_SEC)

ax1.set_ylabel('Gemini\nOnly', fontsize=10, fontweight='bold', labelpad=8, color=COLORS['gemini_only'])
ax2.set_ylabel('CV +\nGemini', fontsize=10, fontweight='bold', labelpad=8, color=COLORS['cv_gemini'])
ax2.set_xlabel('시간 (초)', fontsize=10)

handles = [
    mpatches.Patch(color='#22c55e', alpha=0.5, label=f'실제 화면 전환 ({SCENE_CHANGES_N}회)'),
    mpatches.Patch(color=COLORS['gemini_only'], label=f'Gemini Only 호출 ({go_count}회)'),
    mpatches.Patch(color=COLORS['warning'],     label=f'품질 불량 호출 ({go_on_bad}회)'),
    mpatches.Patch(color=COLORS['cv_gemini'],   label=f'CV+Gemini 호출 ({cv_sent}회, 전환 감지 시만)'),
]
fig.legend(handles=handles, loc='upper center', ncol=4, fontsize=8.5,
           bbox_to_anchor=(0.5, 1.02), framealpha=0.9)

fig.suptitle('API 호출 타임라인 — 언제 전송하는가?',
             fontsize=13, fontweight='bold', y=1.12)
plt.tight_layout()
out = OUTPUT_DIR / 'pipeline-comparison-timeline.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'저장: {out}')

In [ ]:
# ── Chart 4: 전송 프레임 품질 분포 비교 ────────────────────────────────────
# CV+Gemini: 품질 게이트 통과 + 변화 감지 → 전송 품질
cv_sent_quality = df[
    df['passes_quality'] & df['is_scene_change']
]['quality'].values

# Gemini Only: 2초마다 무작위 프레임 — 품질 필터 없음
go_frame_quality = []
for t in go_times:
    nearest = df.iloc[(np.abs(df['time_sec'] - t)).argmin()]
    go_frame_quality.append(nearest['quality'])
go_frame_quality = np.array(go_frame_quality)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
fig.patch.set_facecolor('white')

# KDE 분포
for ax in [ax1, ax2]:
    ax.set_facecolor('#f8f9fd')
    ax.axvline(MIN_Q, color='#6b7280', linestyle='--', linewidth=1.5,
               label=f'최소 품질 임계값 ({MIN_Q})', alpha=0.7)

ax1.hist(go_frame_quality, bins=20, color=COLORS['gemini_only'], alpha=0.75,
         edgecolor='white', label='Gemini Only 전송 프레임')
ax1.set_title(f'Gemini Only — 전송 프레임 품질 분포\n(n={len(go_frame_quality)}, 평균={np.mean(go_frame_quality):.1f})',
              fontsize=10, fontweight='bold')
ax1.set_xlabel('품질 점수 (0~100)')
ax1.set_ylabel('프레임 수')
bad_pct = (go_frame_quality < MIN_Q).mean() * 100
ax1.legend(fontsize=8)
ax1.text(0.97, 0.95, f'{bad_pct:.0f}% 품질 불량',
         transform=ax1.transAxes, ha='right', va='top',
         color=COLORS['gemini_only'], fontsize=11, fontweight='bold')

if len(cv_sent_quality) > 0:
    ax2.hist(cv_sent_quality, bins=max(5, len(cv_sent_quality)//2),
             color=COLORS['cv_gemini'], alpha=0.8,
             edgecolor='white', label='CV+Gemini 전송 프레임')
    ax2.set_title(f'CV + Gemini — 전송 프레임 품질 분포\n(n={len(cv_sent_quality)}, 평균={np.mean(cv_sent_quality):.1f})',
                  fontsize=10, fontweight='bold')
    bad_pct_cv = (cv_sent_quality < MIN_Q).mean() * 100
    ax2.text(0.97, 0.95, f'{bad_pct_cv:.0f}% 품질 불량\n(모두 필터됨)',
             transform=ax2.transAxes, ha='right', va='top',
             color=COLORS['cv_gemini'], fontsize=11, fontweight='bold')
else:
    ax2.text(0.5, 0.5, f'CV+Gemini: {cv_sent}회 전송\n(전부 품질 게이트 통과)',
             transform=ax2.transAxes, ha='center', va='center',
             fontsize=13, color=COLORS['cv_gemini'], fontweight='bold')

ax2.set_xlabel('품질 점수 (0~100)')
ax2.set_ylabel('프레임 수')
ax2.legend(fontsize=8)

fig.suptitle('전송 프레임 품질 분포 — 무엇을 Gemini에 보내는가?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 'pipeline-comparison-quality.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'저장: {out}')

In [ ]:
# ── Chart 5: 히스토그램 FP 비교 — 시나리오별 오감지 감소 효과 ──────────────
# window=1 (기준선) vs window=5 (최적) FP 비교
scenarios_order = ['hand-shake', 'rolling-shutter', 'ios-autofocus', 'lighting', 'normal-change']
scenario_labels = ['손 떨림', 'Rolling\nShutter', 'iOS\n자동초점', '조명\n변화', '정상\n전환']

w1_fp, w5_fp = [], []
for s in scenarios_order:
    sub = hist_df[(hist_df['metric'] == 'HISTCMP_CORREL') & (hist_df['color_space'] == 'GRAY') & (hist_df['scenario'] == s)]
    w1 = sub[sub['window'] == 1]['fp'].mean()
    w5 = sub[sub['window'] == 5]['fp'].mean()
    w1_fp.append(w1 if not np.isnan(w1) else 0)
    w5_fp.append(w5 if not np.isnan(w5) else 0)

x = np.arange(len(scenarios_order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fd')

b1 = ax.bar(x - width/2, w1_fp, width, label='window=1 (기준선)',
            color=COLORS['gemini_only'], alpha=0.8, edgecolor='white')
b2 = ax.bar(x + width/2, w5_fp, width, label='window=5 (최적, 3프레임 연속)',
            color=COLORS['cv_gemini'], alpha=0.85, edgecolor='white')

for bar, v in zip(b1, w1_fp):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.0f}', ha='center', va='bottom', fontsize=9)
for bar, v in zip(b2, w5_fp):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.0f}', ha='center', va='bottom', fontsize=9, color=COLORS['cv_gemini'])

ax.set_xticks(x)
ax.set_xticklabels(scenario_labels, fontsize=10)
ax.set_ylabel('False Positive 수 (오감지)')
ax.set_title('히스토그램 연속 프레임 윈도우 효과 — 오감지 감소\n(HISTCMP_CORREL + Grayscale, 50프레임 동영상 기준)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, max(max(w1_fp), 1) * 1.3)

# 개선율 표시
for i, (v1, v5) in enumerate(zip(w1_fp, w5_fp)):
    if v1 > 0:
        save = (v1 - v5) / v1 * 100
        ax.text(i, max(v1, v5) + max(max(w1_fp), 1)*0.12,
                f'▼{save:.0f}%', ha='center', fontsize=8,
                color=COLORS['cv_gemini'], fontweight='bold')

plt.tight_layout()
out = OUTPUT_DIR / 'pipeline-comparison-fp.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'저장: {out}')

In [ ]:
# ── 요약 테이블 CSV 저장 ──────────────────────────────────────────────────
summary = pd.DataFrame([
    {
        '방식':                  'Gemini Only (매 2초)',
        'API 호출 (60초)':       go_count,
        '품질 불량 호출':        go_on_bad,
        '유효 호출':             go_count - go_on_bad,
        '불필요 호출률':         f'{go_on_bad/go_count:.1%}',
        '추정 비용 ($/hr)':      f'{cost_go:.4f}',
        '실제 화면전환 감지':    f'{min(go_on_change, SCENE_CHANGES_N)}/{SCENE_CHANGES_N}',
        'FP (손떨림)':           f'{w1_fp[0]:.1f}',
    },
    {
        '방식':                  'CV + Gemini (모듈1+2+3)',
        'API 호출 (60초)':       cv_sent,
        '품질 불량 호출':        0,
        '유효 호출':             cv_sent,
        '불필요 호출률':         '0.0%',
        '추정 비용 ($/hr)':      f'{cost_cv:.4f}',
        '실제 화면전환 감지':    f'{min(cv_sent, SCENE_CHANGES_N)}/{SCENE_CHANGES_N}',
        'FP (손떨림)':           f'{w5_fp[0]:.1f}',
    },
])

out_csv = OUTPUT_DIR / 'pipeline-comparison-summary.csv'
summary.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('\n요약 테이블:')
print(summary.to_string(index=False))
print(f'\n저장: {out_csv}')

In [ ]:
# ── 최종 정리 ──────────────────────────────────────────────────────────────
print('=' * 60)
print('CV + Gemini 파이프라인 효율화 — 핵심 수치')
print('=' * 60)
print(f'시나리오: 60초 BIOS 화면 조작 세션 (30fps = {TOTAL_FRAMES:,}프레임)')
print()
print('[모듈 3] 프레임 품질 게이트')
print(f'  거부: {q_fail}프레임 ({q_fail/TOTAL_FRAMES:.1%}) → 흔들림/어두움/과노출')
print(f'  통과: {q_pass}프레임 ({q_pass/TOTAL_FRAMES:.1%})')
print(f'  근거: 실촬영 BIOS 22장 → {REAL_PASS_RATE:.1%} 통과율')
print()
print('[모듈 2] 히스토그램 변화 감지 (CORREL+GRAY+window=5)')
print(f'  3프레임 연속 + 2초 쿨다운 → {cv_sent}회 전송')
print(f'  손 떨림 FP: window=1 기준 {w1_fp[0]:.0f}회 → window=5 {w5_fp[0]:.0f}회')
print()
print('[결과] Gemini API 효율 비교')
print(f'  Gemini Only:  {go_count}회 ({go_on_bad}회 품질불량 포함)')
print(f'  CV + Gemini:  {cv_sent}회 (전부 유효 프레임)')
print(f'  비용 절감: {(1 - cv_sent/go_count)*100:.0f}%')
print()
print('산출 파일:')
for f in OUTPUT_DIR.glob('pipeline-comparison-*.png'):
    print(f'  {f.name}')
print(f'  pipeline-comparison-summary.csv')
print('=' * 60)